# 🎯 [K15_M02] 예약 현황표와 점검 이력표 정리하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K15/k15_m02_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 군수시설·자산관리 (Level 1) / K15 군 시설 정보 검색 |
| **미션** | 미션 02. 예약 현황표와 점검 이력표 정리하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 조건 제어 프롬프트 엔지니어링 및 팩트체크 검증 (HITL) |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.


In [ ]:
# ============================================================================
# [Colab 환경 설정] 데이터 분석 및 프롬프트 테스트 환경 준비
# ============================================================================
!pip install -q pandas numpy tabulate

import pandas as pd
import numpy as np
import json
import re

# 가상 한빛대대 데이터셋 인메모리 로드 (Colab 환경 독립 실행 보장)
schedule_raw = [
    {"facility": "대대 강당", "day": "월", "09-11": "1중대 교육", "11-13": "1중대 교육", "14-16": "훈련 평가", "16-18": "빈", "19-21": "빈"},
    {"facility": "대대 강당", "day": "화", "09-11": "2중대 교육", "11-13": "빈", "14-16": "3중대 교육", "16-18": "빈", "19-21": "합창 동아리"},
    {"facility": "대대 강당", "day": "수", "09-11": "3중대 교육", "11-13": "3중대 교육", "14-16": "빈", "16-18": "정훈 방송", "19-21": "빈"},
    {"facility": "대대 강당", "day": "목", "09-11": "1중대 안전교육", "11-13": "빈", "14-16": "2중대 교육", "16-18": "2중대 교육", "19-21": "빈"},
    {"facility": "대대 강당", "day": "금", "09-11": "월례 조회", "11-13": "점검(음향)", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "영화 상영"},
    {"facility": "대대 강당", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "월", "09-11": "분대장 회의", "11-13": "빈", "14-16": "분대장 회의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "화", "09-11": "빈", "11-13": "인사과 상담", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "수", "09-11": "빈", "11-13": "빈", "14-16": "1중대 토의", "16-18": "1중대 토의", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "목", "09-11": "2중대 토의", "11-13": "2중대 토의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "금", "09-11": "빈", "11-13": "빈", "14-16": "3중대 토의", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "토", "09-11": "독서 모임", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 A", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "월", "09-11": "빈", "11-13": "교육 준비", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "화", "09-11": "점검(빔프로젝터)", "11-13": "점검(빔프로젝터)", "14-16": "점검(빔프로젝터)", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "수", "09-11": "3중대 토의", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "목", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "AI 활용 스터디", "19-21": "AI 활용 스터디"},
    {"facility": "세미나실 B", "day": "금", "09-11": "대대 간부 회의", "11-13": "대대 간부 회의", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "토", "09-11": "빈", "11-13": "자격증 스터디", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "세미나실 B", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "월", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "화", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "수", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "소독(이용 불가)", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "목", "09-11": "체력검정 연습(3중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "3중대 단체 운동"},
    {"facility": "체력단련실(단체)", "day": "금", "09-11": "체력검정 연습(1·2중대)", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "토", "09-11": "빈", "11-13": "빈", "14-16": "대대 체육 연습", "16-18": "대대 체육 연습", "19-21": "빈"},
    {"facility": "체력단련실(단체)", "day": "일", "09-11": "빈", "11-13": "빈", "14-16": "빈", "16-18": "빈", "19-21": "빈"}
]

maintenance_raw = [
    {"id": "H01", "date": "08-03", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H02", "date": "08-10", "facility": "대대 강당", "type": "정기점검", "content": "음향 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H03", "date": "08-17", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H04", "date": "08-24", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H05", "date": "08-31", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H06", "date": "09-02", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 전원 꺼짐", "dept": "군수과", "status": "완료", "days": 3},
    {"id": "H07", "date": "09-07", "facility": "체력단련실", "type": "정기점검", "content": "운동 기구 안전 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H08", "date": "09-09", "facility": "사이버지식정보방", "type": "고장신고", "content": "PC 3대 부팅 불가", "dept": "정보통신실", "status": "완료", "days": 2},
    {"id": "H09", "date": "09-14", "facility": "대대 강당", "type": "고장신고", "content": "출입문 잠금장치 불량", "dept": "군수과", "status": "완료", "days": 5},
    {"id": "H10", "date": "09-16", "facility": "세미나실 A", "type": "정기점검", "content": "환기 설비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H11", "date": "09-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "완료", "days": 6},
    {"id": "H12", "date": "09-23", "facility": "세미나실 B", "type": "정기점검", "content": "영상 장비 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H13", "date": "09-28", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "완료", "days": 3},
    {"id": "H14", "date": "10-01", "facility": "세미나실 A", "type": "고장신고", "content": "난방기 소음", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H15", "date": "10-05", "facility": "사이버지식정보방", "type": "정기점검", "content": "PC·네트워크 점검", "dept": "정보통신실", "status": "완료", "days": 1},
    {"id": "H16", "date": "10-08", "facility": "세미나실 B", "type": "고장신고", "content": "빔프로젝터 화면 깜빡임", "dept": "군수과", "status": "완료", "days": 4},
    {"id": "H17", "date": "10-13", "facility": "체력단련실", "type": "고장신고", "content": "샤워실 배수 불량", "dept": "군수과", "status": "완료", "days": 2},
    {"id": "H18", "date": "10-15", "facility": "대대 강당", "type": "정기점검", "content": "조명·음향 점검", "dept": "군수과", "status": "완료", "days": 1},
    {"id": "H19", "date": "10-19", "facility": "세미나실 B", "type": "고장신고", "content": "에어컨 냉기 약함", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H20", "date": "10-21", "facility": "체력단련실", "type": "고장신고", "content": "러닝머신 1호 벨트 이탈", "dept": "군수과", "status": "처리 중", "days": np.nan},
    {"id": "H21", "date": "10-22", "facility": "대대 강당", "type": "고장신고", "content": "무선 마이크 수신 불량", "dept": "정보통신실", "status": "접수", "days": np.nan},
    {"id": "H22", "date": "10-23", "facility": "세미나실 A", "type": "정기점검", "content": "난방기 점검", "dept": "군수과", "status": "완료", "days": 1}
]

df_schedule = pd.DataFrame(schedule_raw)
df_maintenance = pd.DataFrame(maintenance_raw)

print("✅ [K15] 가상 한빛대대 실습 환경 및 검산용 데이터프레임 로드 완료!")
print(f"- 1주 시설 예약 현황표: {len(df_schedule)}행 (4개 시설 × 7일)")
print(f"- 시설 점검 및 고장 이력: {len(df_maintenance)}건 (정기점검 7건, 고장신고 15건)")


---

## 🎯 미션 02. 예약 현황표와 점검 이력표 정리하기
**소요시간**: 약 1.5시간 | **난이도**: 초급 (Level 1) | **유형**: AI 활용형

### 📋 미션 개요 & 핵심 학습 목표
**[미션 개요]**
<p>생성형 AI가 데이터를 왜곡 없이 검색할 수 있도록 비정형 시설 정보를 체계적인 참조 데이터셋으로 전처리합니다. 4개 시설의 1주 시간대별 예약 현황 28행과 점검·고장 이력 22건을 시설·요일·시간대·상태 항목으로 정형화하고, 점검 및 소독 등 예약 불가 구간을 명확히 분리 표기합니다. 이를 통해 AI 환각을 방지하고 정확한 근거 기반 검색을 위한 데이터 기초를 마련합니다.</p>

**[학습 목표 및 체득 역량]**
<p>4개 시설의 1주 시간대별 예약 현황(28행)과 점검·고장 이력(22건)을 시설·요일·시간대·상태 항목이 분명한 표로 정리하고 이용 불가 구간을 명확히 구분합니다. <strong>이를 통해 AI가 왜곡 없이 참조할 수 있도록 비정형 행정 문서를 규칙화된 데이터셋으로 전처리하는 데이터 구조화 역량을 익힙니다.</strong></p>

### 🛠️ 세부 실습 활동 (GuideSteps)
#### 🔹 Step 1: 예약 현황표 28행을 확인한다.
<h4>이 단계의 역할</h4><p>가상 한빛대대의 4개 시설(대대 강당, 세미나실 A, 세미나실 B, 체력단련실) 1주일 치(월~일) 예약 현황표 28행 데이터를 검토하고 구조화합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>실습 자료의 시설 예약 현황표 28행을 열람하고 시설명, 요일, 시간대(09-11, 11-13, 14-16, 16-18, 19-21) 구조를 확인합니다.</p></li>
<li><p>'빈'으로 표기된 칸과 실제 부대 일정(중대 교육, 훈련 평가 등)이 등록된 칸을 구분합니다.</p></li>
<li><p>표의 행(Row) 수가 4개 시설 × 7일 = 총 28행으로 누락 없이 작성되었는지 확인합니다.</p></li>
</ol>
<h4>💡 AI 프롬프트 구조화 예시</h4><blockquote><p><strong>[표 구조화 질의]:</strong> "아래 4개 시설의 1주일 예약 현황표 텍스트 데이터를 AI가 행과 열을 오인하지 않도록 마크다운 테이블(Markdown Table) 형식으로 정렬하고 총 행 수가 28행인지 확인해 줘: [예약 현황 원문]"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>행 수 일치:</strong> 강당 7행, 세미나실 A 7행, 세미나실 B 7행, 체력단련실 7행 총 28행이 정확한가?</li>
<li><strong>시간대 완전성:</strong> 09-11부터 19-21까지 5개 시간대 열이 왜곡 없이 유지되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>예약 현황표 28행 완전성 확인 및 마크다운 정렬 완료</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>정리된 예약 현황표 28행 검증본</p></li></ul>

#### 🔹 Step 2: 점검 이력 22건을 확인한다.
<h4>이 단계의 역할</h4><p>8월부터 10월까지 축적된 가상 한빛대대 시설 점검 및 고장 신고 이력 22건(H01~H22)의 필드 구조를 정리합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>점검 이력표 원문에서 번호, 일자, 시설, 구분(정기점검/고장신고), 내용, 처리부서, 상태, 소요일 항목을 점검합니다.</p></li>
<li><p>데이터 건수가 H01부터 H22까지 총 22건이 맞는지 카운트합니다.</p></li>
<li><p>미완료 건(처리 중, 접수)의 소요일 표기('-')가 정상적으로 기재되어 있는지 확인합니다.</p></li>
</ol>
<h4>💡 AI 프롬프트 검증 예시</h4><blockquote><p><strong>[이력 데이터 검증 질의]:</strong> "제공된 시설 점검·고장 이력 데이터 22건에서 누락된 행 번호(H01~H22)가 있는지 검사하고, 각 레코드의 필수 8개 컬럼이 온전한지 테이블로 요약해 줘: [점검 이력 원문]"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>총 건수 검증:</strong> H01부터 H22까지 22건 전체가 정확히 존재하는가?</li>
<li><strong>속성 일치:</strong> 정기점검 7건, 고장신고 15건 구분이 원문과 오차 없이 일치하는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>점검 이력 22건 일련번호 및 컬럼 정합성 100% 검증</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>정리된 시설 점검·고장 이력표 22건</p></li></ul>

#### 🔹 Step 3: 점검·소독 칸을 이용 불가로 표시한다.
<h4>이 단계의 역할</h4><p>예약 현황표에서 실제 예약이 없더라도 점검이나 소독으로 인해 이용할 수 없는 '이용 불가' 칸을 식별하여 강조 표시합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>예약표에서 '점검' 또는 '소독'이 기재된 칸을 전수 조사합니다.</p></li>
<li><p>강당 금요일 11-13(음향 점검), 세미나실 B 화요일 09-16(빔프로젝터 점검 3개 슬롯), 체력단련실 수요일 14-16(소독)을 적출합니다.</p></li>
<li><p>해당 칸들을 '이용 불가(점검/소독)'로 주석 처리하거나 붉은색/별도 라벨로 마킹합니다.</p></li>
</ol>
<h4>💡 AI 프롬프트 필터링 예시</h4><blockquote><p><strong>[이용 불가 슬롯 추출 질의]:</strong> "아래 예약 현황표에서 부대 교육이나 회의 예약이 아니지만, 설비 점검이나 방역 소독으로 인해 장병들이 절대 이용할 수 없는 칸(시설, 요일, 시간대)을 모두 찾아 표로 정리해 줘."</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>식별 완전성:</strong> 강당 1건, 세미나실 B 3개 슬롯(09-11, 11-13, 14-16), 체력단련실 1건 총 5개 슬롯이 정확히 식별되었는가?</li>
<li><strong>빈 칸과의 구별:</strong> '빈' 칸과 '점검/소독' 칸이 명확히 분리 표기되었는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>이용 불가 칸 5개 슬롯 정확 식별 및 마킹 완료</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>이용 불가 칸 식별 및 표시 완료된 예약 현황표</p></li></ul>

#### 🔹 Step 4: 심화 도전 과제 및 최적화
<h4>이 단계의 역할</h4><p>시설별로 1주일간 이용 가능한 순수 '빈' 칸의 총 개수를 직접 세고 AI 검산 결과와 대조합니다.</p>
<h4>해야 할 일</h4><ol>
<li><p>4개 시설별로 '빈'으로 표시된 슬롯 수를 요일별로 수작업 집계합니다.</p></li>
<li><p>AI에게 각 시설별 빈 슬롯 수 집계를 질의하고 본인의 수작업 카운트와 비교합니다.</p></li>
<li><p>집계 불일치가 발생할 경우 '점검/소독' 칸을 빈 칸으로 잘못 셌는지 원인을 규명합니다.</p></li>
</ol>
<h4>💡 AI 집계 검산 질의 예시</h4><blockquote><p><strong>[빈 슬롯 카운트 질의]:</strong> "제공된 예약표에서 오직 '빈'이라고 적힌 슬롯만 세어 시설별(대대 강당, 세미나실 A, 세미나실 B, 체력단련실) 주간 빈 시간 합계를 숫자로 알려줘. (점검/소독은 절대 빈 칸에 포함하지 말 것)"</p></blockquote>
<h4>🔍 검증 포인트 (원문 대조 및 팩트체크)</h4><ul>
<li><strong>수치 검산:</strong> 강당, 세미나실 A, 세미나실 B, 체력단련실의 실제 빈 슬롯 합계가 일치하는가?</li>
<li><strong>오류 배제:</strong> 점검 칸이 카운트에 합산되는 오류가 발생하지 않았는가?</li>
</ul>
<h4>핵심 확인 지표</h4><blockquote><p>시설별 주간 순수 빈 칸 수 집계표 및 수작업 검산 일치율 100%</p></blockquote>
<h4>학습자가 제출해야 하는 내용</h4><ul><li><p>시설별 가용 빈 슬롯 수 집계표 및 자가 검산 메모</p></li></ul>

### 📑 실습 작성 서식 및 체크리스트 (Sample Template)

## 1. 정제된 예약 현황표 구조 점검 (28행)
- [x] 시설 4개 (대대 강당, 세미나실 A, 세미나실 B, 체력단련실) × 7일 = 총 28행 확인
- [x] 시간대 5개 열 (09-11, 11-13, 14-16, 16-18, 19-21) 누락 없음 확인

## 2. 이용 불가 칸 (점검 및 소독) 식별 및 표시표
| 시설명 | 요일 | 시간대 | 원본 표기 내용 | 이용 불가 사유 |
| :--- | :--- | :--- | :--- | :--- |
| 대대 강당 | 금 | 11-13 | 점검(음향) | 음향 설비 점검으로 인한 이용 불가 |
| 세미나실 B | 화 | 09-11 | 점검(빔프로젝터) | 빔프로젝터 점검으로 인한 이용 불가 |
| 세미나실 B | 화 | 11-13 | 점검(빔프로젝터) | 빔프로젝터 점검으로 인한 이용 불가 |
| 세미나실 B | 화 | 14-16 | 점검(빔프로젝터) | 빔프로젝터 점검으로 인한 이용 불가 |
| 체력단련실(단체) | 수 | 14-16 | 소독(이용 불가) | 방역 소독 작업으로 인한 이용 불가 |

## 3. 시설별 주간 순수 '빈' 슬롯 수 집계표
| 시설명 | 월 | 화 | 수 | 목 | 금 | 토 | 일 | 주간 빈 슬롯 합계 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| 대대 강당 | 2 | 2 | 2 | 2 | 2 | 4 | 5 | 19개 |
| 세미나실 A | 3 | 4 | 3 | 3 | 4 | 4 | 5 | 26개 |
| 세미나실 B | 4 | 2 | 4 | 3 | 3 | 4 | 5 | 25개 |
| 체력단련실(단체)| 4 | 4 | 3 | 3 | 4 | 3 | 5 | 26개 |
| **전체 합계** | | | | | | | | **96개** |
*(주의: '점검' 4개 슬롯 및 '소독' 1개 슬롯은 빈 슬롯에서 반드시 제외)*

### 🔍 평가 기준 (PASS 조건)
<h4>미션 판정</h4><p><strong>평가 구분:</strong> 필수</p><p><strong>핵심 평가 기준:</strong> [필수] 정보 구조화·데이터 전처리 — 28행 예약 현황표와 22건 점검 이력을 정형화하고 이용 불가 칸(점검·소독)을 명확히 분리했는가? (PASS / FAIL)</p><h4>미션 PASS 조건</h4><ul><li><p>예약 현황표 28행(4개 시설 × 7일) 및 점검 이력 22건 완전성 충족</p></li><li><p>점검 및 소독 5개 슬롯의 이용 불가 마킹 및 순수 빈 칸 집계 정확성 확보</p></li><li><p>교육용 가상 데이터 활용 및 군사보안 수칙 엄수</p></li></ul><h4>채점자 판정 메모</h4><blockquote><p>AI의 오작동을 막기 위한 핵심 전처리인 '점검'과 '소독' 슬롯을 '빈' 슬롯과 혼동 없이 분리했는지를 집중 검증한다.</p></blockquote>


In [ ]:
# [미션 02] 프롬프트 및 실습 결과 기록
mission_02_prompt = """
# [TODO] 28행 예약 현황표와 22건 점검 이력에서 '점검' 및 '소독' 칸을 이용 불가로 분리 식별하고,
# 시설별 주간 순수 '빈' 슬롯 수를 정형화하여 집계하도록 유도하는 프롬프트를 작성하세요.
"""

mission_02_result = """
# [TODO] AI 질의 응답 결과 및 이용 불가 칸(5개) 식별표, 주간 빈 슬롯 집계표를 여기에 기록하세요.
"""

print(f"미션 02 실습 기록 완료: 프롬프트 {len(mission_02_prompt.strip())}자, 결과 {len(mission_02_result.strip())}자")


In [ ]:
# [데이터 교차 검산 - 미션2] 시설별 주간 순수 '빈' 슬롯 및 '이용 불가' 슬롯 집계
time_cols = ["09-11", "11-13", "14-16", "16-18", "19-21"]

def count_slots(row, target):
    return sum(1 for c in time_cols if target in str(row[c]))

df_counts = df_schedule.copy()
df_counts['빈_칸'] = df_counts.apply(lambda r: count_slots(r, "빈"), axis=1)
df_counts['점검_칸'] = df_counts.apply(lambda r: count_slots(r, "점검"), axis=1)
df_counts['소독_칸'] = df_counts.apply(lambda r: count_slots(r, "소독"), axis=1)

summary = df_counts.groupby("facility")[['빈_칸', '점검_칸', '소독_칸']].sum()
print("=== 시설별 주간 슬롯 집계 결과 ===")
print(summary)
print(f"\n총 빈 슬롯: {summary['빈_칸'].sum()}개 (점검 {summary['점검_칸'].sum()}개, 소독 {summary['소독_칸'].sum()}개 제외 완료)")
